# CSBS BS metrics by examiner

**Study:** NANO Lab Assessments & Double Data Entry (REDCap PID 5484)
**Output:** `csbsbs_metrics_by_examiner.xlsx` with the mean and SD of the three CSBS DP Behavior Sample composite raw scores (Social, Speech, Symbolic) for each examiner, visit and diagnostic group.

How the notebook keeps the numbers right:

1. It connects with the PID 5484 token from `.env` and stops if the API reports any other project.
2. It finds every CSBS BS field in the live data dictionary by its **label** and checks that each score formula in REDCap reads the right inputs.
3. It keeps only the **merged** double-data-entry record of each visit (`5001`, not `5001--1` or `5001--2`).
4. It re-scores every visit from the worksheet boxes and compares the result with the scores stored in REDCap.
5. It saves the metrics table with a QC sheet and a field map. The notebook only reads from REDCap and never writes to it.

In [1]:
import difflib
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display
from openpyxl.styles import Alignment, Font

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 120)

# The repo root is the first folder (this one or a parent) that holds the git-ignored .env with the API tokens.
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("No .env file in this folder or its parents. It must hold the REDCap API tokens.")
load_dotenv(REPO_ROOT / ".env")

API_URL = os.getenv("REDCAP_API_URL", "https://redcap.research.sc.edu/api/")
OUTPUT_XLSX = REPO_ROOT / "IPSA" / "csbsbs_metrics_by_examiner.xlsx"
AUDIT_CSV = REPO_ROOT / "IPSA" / "csbsbs_audit_secure" / "csbsbs_visit_audit.csv"  # participant-level, git-ignored

STUDY = {"pid": "5484", "token_env": "TOKEN_NANO_LAB_ASSESSMENTS_5484"}
# PID 5484 leaves demo_status blank for most children. Those children take the same field from
# NANO Study Surveys (PID 4218), matched on child ID. Set to False to use PID 5484 only.
FILL_GROUP_FROM_NANO_SURVEYS = True
NANO_SURVEYS = {"pid": "4218", "token_env": "NANO_API_TOKEN"}
# Visits that REDCap marks "Check before using these scores" are kept unless this is True.
EXCLUDE_REDCAP_FLAGGED = False

qc = []  # one row per check, saved to the "QC Checks" sheet


def check(name, result, status="OK"):
    qc.append({"Check": name, "Result": result, "Status": status})


def redcap(token, content, **params):
    """Send one read-only export request to the REDCap API and return the parsed JSON."""
    payload = {"token": token, "content": content, "format": "json", "returnFormat": "json"}
    for key, value in params.items():
        if isinstance(value, (list, tuple)):
            payload.update({f"{key}[{i}]": item for i, item in enumerate(value)})
        else:
            payload[key] = value
    response = requests.post(API_URL, data=payload, timeout=120)
    if response.status_code != 200:
        raise RuntimeError(f"REDCap '{content}' export failed (HTTP {response.status_code}): {response.text[:300]}")
    return response.json()


def connect(spec):
    """Read the token named in `spec` from .env and confirm the API says it opens the expected project."""
    token = os.getenv(spec["token_env"], "").strip()
    if not token:
        raise RuntimeError(f"Add {spec['token_env']}=<API token for PID {spec['pid']}> to {REPO_ROOT / '.env'}")
    info = redcap(token, "project")
    if str(info["project_id"]) != spec["pid"]:
        raise RuntimeError(
            f"{spec['token_env']} opens PID {info['project_id']} ({info['project_title']}), not PID {spec['pid']}."
        )
    return token, info


TOKEN, project = connect(STUDY)
check("REDCap project", f"PID {project['project_id']}: {project['project_title']} (token {STUDY['token_env']} in .env)")
print(f"Connected to PID {project['project_id']}: {project['project_title']}")

Connected to PID 5484: NANO Lab Assessments & Double Data Entry


## 1. Find the CSBS BS fields by label

Every field is looked up by its label in the live data dictionary, so a renamed variable or a second copy of a field stops the run instead of being read silently. A label has to match exactly one field on the form.

| What | Label it must match | Today's variable |
|---|---|---|
| Examiner | `Scored by ...` (the old form called it "Examiner ... only enter the scorer's name") | `csbsbs_examiner` |
| 22 scale boxes | `Scale 1:` to `Scale 20:`, plus `Scale 16, object / person / body ...` | `csbsbs_scale1` ... |
| 7 cluster scores | `<cluster> weighted raw score` | `csbsbs_emotionraw` ... |
| 3 composites and the total | `<Social / Speech / Symbolic> composite weighted raw score`, `Total weighted raw score` | `csbsbs_socialcompositecalc` ... |

Then each calculated score's REDCap formula must read exactly the inputs the CSBS DP scoring rules call for. For example, the Social composite must read Emotion and Eye Gaze, Communication and Gestures, and nothing else.

In [2]:
metadata = pd.DataFrame(redcap(TOKEN, "metadata"))
metadata["label"] = metadata["field_label"].str.replace(r"<[^>]+>", "", regex=True).str.strip()
meta = metadata.set_index("field_name")
instruments = pd.DataFrame(redcap(TOKEN, "instrument"))
events = pd.DataFrame(redcap(TOKEN, "event"))
form_events = pd.DataFrame(redcap(TOKEN, "formEventMapping"))
RECORD_ID = metadata["field_name"].iloc[0]
EVENT_LABEL = dict(zip(events["unique_event_name"], events["event_name"]))
VISIT_ORDER = list(events["event_name"])

csbs_forms = instruments[instruments["instrument_label"].str.match(r"CSBS\s*BS\b", case=False)]
current = csbs_forms[~csbs_forms["instrument_label"].str.contains(r"\bold\b", case=False)]
if len(current) != 1:
    raise ValueError(f"Expected one current CSBS BS instrument, found {csbs_forms['instrument_label'].tolist()}")
FORM, FORM_LABEL = current.iloc[0][["instrument_name", "instrument_label"]]
LEGACY_FORMS = [name for name in csbs_forms["instrument_name"] if name != FORM]
CSBS_EVENTS = form_events.loc[form_events["form"] == FORM, "unique_event_name"].tolist()
form_fields = metadata[(metadata["form_name"] == FORM) & (metadata["field_type"] != "descriptive")]


def field_by_label(pattern, fields=form_fields):
    """The one field whose label matches `pattern`. Stops with the candidates if none or several match."""
    hits = fields.loc[fields["label"].str.contains(pattern, case=False, regex=True), "field_name"].tolist()
    if len(hits) != 1:
        raise ValueError(f"Label /{pattern}/ matched {len(hits)} fields: {hits}")
    return hits[0]


def parse_choices(text):
    """'1, TD Sib | 2, ASD Sib' -> {'1': 'TD Sib', '2': 'ASD Sib'}"""
    pairs = (item.split(",", 1) for item in str(text).split("|"))
    return {code.strip(): label.strip() for code, label in pairs}


def choices(field):
    return parse_choices(meta.at[field, "select_choices_or_calculations"])


# CSBS DP Behavior Sample scoring: a cluster is a weighted sum of its scales (rounded, .5 up, where
# the manual says so), a composite is the sum of its clusters and the total is the sum of the composites.
CLUSTERS = {  # cluster: ({scale: weight}, rounded)
    "Emotion and Eye Gaze": ({"1": 1, "2": 1, "3": 3}, False),
    "Communication": ({"4": 1 / 3, "5": 1, "6": 1, "7": 1}, True),
    "Gestures": ({"8": 2, "9": 1}, False),
    "Sounds": ({"10": 1, "11": 2}, False),
    "Words": ({"12": 1, "13": 1 / 2, "14": 1, "15": 1}, True),
    "Understanding": ({"16 object": 3, "16 person": 3, "16 body": 3}, False),
    "Object Use": ({"17": 1, "18": 1, "19": 1, "20": 1}, False),
}
COMPOSITES = {
    "Social": ["Emotion and Eye Gaze", "Communication", "Gestures"],
    "Speech": ["Sounds", "Words"],
    "Symbolic": ["Understanding", "Object Use"],
}

SCALE = {}  # "1" ... "20", "16 object", "16 person", "16 body" -> field
for field, label in zip(form_fields["field_name"], form_fields["label"]):
    match = re.match(r"Scale (\d+)(?::|, (\w+))", label)
    if match:
        key = " ".join(filter(None, match.groups()))
        if key in SCALE:
            raise ValueError(f"Two fields are labelled Scale {key}: {SCALE[key]} and {field}")
        SCALE[key] = field
missing = {s for weights, _ in CLUSTERS.values() for s in weights} - set(SCALE)
if missing:
    raise ValueError(f"No field labelled Scale {sorted(missing)} on {FORM}")

CLUSTER = {name: field_by_label(rf"^{name} weighted raw score") for name in CLUSTERS}
COMPOSITE = {name: field_by_label(rf"^{name} composite weighted raw score") for name in COMPOSITES}
TOTAL = field_by_label(r"^Total weighted raw score")
EXAMINER = field_by_label(r"^(?:Scored by|Examiner)\b")
WORKSHEET_ENTERED = field_by_label(r"^Worksheet entered")
NEEDS_REVIEW = field_by_label(r"^Needs review \(1 = red status box")
STATUS = field_by_label(r"^Status$", metadata[metadata["field_type"] == "checkbox"])
STATUS_EVENT = form_events.loc[form_events["form"] == meta.at[STATUS, "form_name"], "unique_event_name"].iloc[0]


def formula_inputs(field):
    """Fields a REDCap calc reads, e.g. [csbsbs_scoring_3(2)] -> csbsbs_scoring_3."""
    return set(re.findall(r"\[([a-z][a-z0-9_]*)(?:\([^)]*\))?\]", meta.at[field, "select_choices_or_calculations"]))


calc_scales = {key: field for key, field in SCALE.items() if meta.at[field, "field_type"] == "calc"}
expected_inputs = {CLUSTER[name]: {SCALE[s] for s in weights} for name, (weights, _) in CLUSTERS.items()}
expected_inputs |= {COMPOSITE[name]: {CLUSTER[c] for c in parts} for name, parts in COMPOSITES.items()}
expected_inputs[TOTAL] = set(COMPOSITE.values())


def wiring(field):
    if field in expected_inputs:
        reads = formula_inputs(field)
        return "Yes" if reads == expected_inputs[field] else f"NO: reads {sorted(reads)}, needs {sorted(expected_inputs[field])}"
    if field in calc_scales.values():
        return "Checked by re-scoring the worksheet (section 3)"
    return "Helper flag, not a score" if meta.at[field, "field_type"] == "calc" else "Entered on the form (no formula)"


roles = (
    [("Examiner", EXAMINER), ("Diagnostic group", STATUS), ("Worksheet entered (helper)", WORKSHEET_ENTERED),
     ("REDCap 'check before using these scores' flag", NEEDS_REVIEW)]
    + [(f"Scale {key}", field) for key, field in SCALE.items()]
    + [(f"{name} cluster", field) for name, field in CLUSTER.items()]
    + [(f"{name} composite", field) for name, field in COMPOSITE.items()]
    + [("Total raw score", TOTAL)]
)
field_map = pd.DataFrame(
    [
        {
            "Role": role,
            "Field variable": field,
            "Field label (live REDCap)": meta.at[field, "label"],
            "Field type": meta.at[field, "field_type"],
            "Instrument": meta.at[field, "form_name"],
            "Formula reads the right inputs": wiring(field),
            "REDCap formula": (meta.at[field, "select_choices_or_calculations"] if meta.at[field, "field_type"] == "calc"
                               else "none (entered on the form)"),
        }
        for role, field in roles
    ]
)
ROLE_OF = dict(zip(field_map["Field variable"], field_map["Role"]))

bad = field_map[field_map["Formula reads the right inputs"].str.startswith("NO")]
if len(bad):
    display(bad)
    raise ValueError("A CSBS BS score formula in REDCap reads the wrong inputs. Fix it before reporting metrics.")

check("CSBS BS instrument", f"{FORM} ('{FORM_LABEL}') at {', '.join(EVENT_LABEL[e] for e in CSBS_EVENTS)}")
check("Fields found by label", f"{len(field_map)} fields; every label matched exactly one field")
check("Score formulas read the right inputs", f"{len(expected_inputs)} of {len(expected_inputs)} cluster, composite and total formulas")
print(f"Instrument: {FORM} ('{FORM_LABEL}') at {[EVENT_LABEL[e] for e in CSBS_EVENTS]}")
print(f"Diagnostic group field: {STATUS} at {EVENT_LABEL[STATUS_EVENT]}, choices {choices(STATUS)}")
assert not field_map.eq("").any().any(), "blank cell in the field map"
display(field_map.loc[~field_map["Role"].str.startswith("Scale"),
                      ["Role", "Field variable", "Field label (live REDCap)", "Formula reads the right inputs"]])

Instrument: csbs_bs ('CSBS BS') at ['9 Months', '12 Months']
Diagnostic group field: demo_status at Consent, choices {'1': 'TD Sib', '2': 'ASD Sib', '3': 'PT', '4': 'Other'}


,Role,Field variable,Field label (live REDCap),Formula reads the right inputs
0,Examiner,csbsbs_examiner,Scored by (scorer's name),Entered on the form (no formula)
1,Diagnostic group,demo_status,Status,Entered on the form (no formula)
2,Worksheet entered (helper),csbsbs_ws_entered,"Worksheet entered (helper: 1 once any worksheet box, rating or cou...","Helper flag, not a score"
3,REDCap 'check before using these scores' flag,csbsbs_qc_flag,"Needs review (1 = red status box shown, 0 = green; helper for Data...","Helper flag, not a score"
26,Emotion and Eye Gaze cluster,csbsbs_emotionraw,Emotion and Eye Gaze weighted raw score (Scale 1 + Scale 2 + Scale...,Yes
27,Communication cluster,csbsbs_comraw,Communication weighted raw score (Scale 4 / 3 + Scale 5 + Scale 6 ...,Yes
28,Gestures cluster,csbsbs_gesraw,Gestures weighted raw score (Scale 8 x 2 + Scale 9),Yes
29,Sounds cluster,csbsbs_soundsraw,Sounds weighted raw score (Scale 10 + Scale 11 x 2),Yes
30,Words cluster,csbsbs_wordsraw,Words weighted raw score (Scale 12 + Scale 13 / 2 + Scale 14 + Sca...,Yes
31,Understanding cluster,csbsbs_underraw,Understanding weighted raw score ((object + person + body part nam...,Yes


## 2. Pull the visits and keep only the merged record

Double data entry stores each visit up to three times: two typists enter `5001--1` and `5001--2`, and the Data Comparison Tool merges them into `5001`, the validated copy. Counting all three would triple-count the child.

Example: `5001`, `5001--1` and `5001--2` at 9 Months count as **one** visit, read from `5001`.

A visit is **scored** when all three composites have a value. Forms with a blank composite (for example a missing Scale 13 count) and visits typed in a DDE copy but not merged yet are left out and counted in the QC sheet.

In [3]:
item_fields = set().union(*(formula_inputs(field) for field in calc_scales.values())) - {WORKSHEET_ENTERED}
score_fields = [*SCALE.values(), *CLUSTER.values(), *COMPOSITE.values(), TOTAL]
export_fields = [RECORD_ID, STATUS, EXAMINER, WORKSHEET_ENTERED, NEEDS_REVIEW, f"{FORM}_complete",
                 *score_fields, *sorted(item_fields)]

rows = pd.DataFrame(redcap(TOKEN, "record", type="flat", rawOrLabel="raw", rawOrLabelHeaders="raw",
                           fields=export_fields, events=[STATUS_EVENT, *CSBS_EVENTS]))
rows["child_id"] = rows[RECORD_ID].str.replace(r"--\d+$", "", regex=True)
rows["dde_copy"] = rows[RECORD_ID].str.extract(r"--(\d+)$", expand=False)
for field in score_fields:
    rows[field] = pd.to_numeric(rows[field], errors="coerce")

csbs = rows[rows["redcap_event_name"].isin(CSBS_EVENTS)].copy()
csbs["Visit"] = csbs["redcap_event_name"].map(EVENT_LABEL)
csbs["scored"] = csbs[list(COMPOSITE.values())].notna().all(axis=1)
merged = csbs[csbs["dde_copy"].isna()]
work = merged[merged[WORKSHEET_ENTERED].eq("1")].copy()  # REDCap scores a worksheet once any box is entered

in_copy = csbs.loc[csbs["dde_copy"].notna() & csbs["scored"], ["child_id", "redcap_event_name", "Visit"]].drop_duplicates()
pending = in_copy.merge(merged.loc[merged["scored"], ["child_id", "redcap_event_name"]], how="left", indicator=True)
pending = pending[pending["_merge"] == "left_only"].drop(columns="_merge")

legacy_visits = 0
if LEGACY_FORMS:
    legacy = pd.DataFrame(redcap(TOKEN, "record", type="flat", rawOrLabel="raw", fields=[RECORD_ID],
                                 forms=LEGACY_FORMS, events=CSBS_EVENTS))
    values = [c for c in legacy.columns if c not in (RECORD_ID, "redcap_event_name") and "___" not in c and not c.endswith("_complete")]
    legacy_visits = int(legacy[values].ne("").any(axis=1).sum())

n_partial = int((~work["scored"]).sum())
check("Double data entry copies left out", f"{csbs['dde_copy'].notna().sum()} rows (--1 / --2); each visit is read from its merged record", "INFO")
check("Visits with CSBS BS worksheet data (merged records)", f"{len(work)} visits, {work['child_id'].nunique()} children", "INFO")
check("Visits scored on all three composites", f"{int(work['scored'].sum())} visits")
check("Partly scored visits left out (a composite is blank)", f"{n_partial} visits: finish the worksheet in REDCap", "REVIEW" if n_partial else "OK")
check("Visits scored only in a DDE copy, not merged yet (left out)", f"{len(pending)} visits: merge them in the Data Comparison Tool", "REVIEW" if len(pending) else "OK")
check("Legacy CSBS BS instrument", f"{legacy_visits} visits hold data on {', '.join(LEGACY_FORMS) or 'none'}", "REVIEW" if legacy_visits else "OK")

print(f"{len(csbs)} CSBS BS visit rows exported: {csbs['dde_copy'].notna().sum()} DDE copies left out, "
      f"{len(merged)} merged records kept, {len(work)} of them with worksheet data.")
display(pd.crosstab(work["Visit"], work["scored"].map({True: "scored", False: "partly scored"}), margins=True))
print(f"Scored only in a DDE copy, not merged yet: {len(pending)} visits. Legacy-form visits: {legacy_visits}.")

850 CSBS BS visit rows exported: 544 DDE copies left out, 306 merged records kept, 231 of them with worksheet data.


scored,partly scored,scored,All
Visit,,,
12 Months,9,99,108
9 Months,9,114,123
All,18,213,231


Scored only in a DDE copy, not merged yet: 32 visits. Legacy-form visits: 0.


## 3. Re-score every visit from the worksheet boxes

Each scale is recounted from the boxes on the worksheet, then the clusters and composites are rebuilt with the CSBS DP weights:

| Score | Formula |
|---|---|
| Emotion and Eye Gaze | S1 + S2 + 3 × S3 (S3 counts Balloon and Books only) |
| Communication | round(S4 ÷ 3 + S5 + S6 + S7), .5 rounds up |
| Gestures | 2 × S8 + S9 |
| Sounds | S10 + 2 × S11 (/b/ or /p/, /d/ or /t/, /g/ or /k/ count once) |
| Words | round(S12 + S13 ÷ 2 + S14 + S15), .5 rounds up |
| Understanding | 3 × (object + person + body part names) |
| Object Use | S17 + S18 + S19 + S20 (S17 adds 1 if an "other action" is written) |
| **Social** = Emotion and Eye Gaze + Communication + Gestures | **Speech** = Sounds + Words | **Symbolic** = Understanding + Object Use |

Worked example: the six Item 4 ratings add up to S4 = 10, and S5 = 2, S6 = 1, S7 = 1, so Communication = round(3.33 + 4) = 7. With S1 = 4, S2 = 3 and S3 = 1, Emotion and Eye Gaze = 4 + 3 + 3 = 10. With S8 = 3 and S9 = 2, Gestures = 8. Social = 10 + 7 + 8 = **25**.

The table below should show 0 mismatches. A mismatch means REDCap holds a stale calculation (run Data Quality rule H) or a formula has drifted from the scoring rules.

In [4]:
def ticked(field, codes):
    return work[[f"{field}___{code}" for code in codes]].eq("1").sum(axis=1)


def rescore_scale(key):
    """Recount one calculated scale from the worksheet items its REDCap formula reads."""
    inputs = sorted(formula_inputs(SCALE[key]) - {WORKSHEET_ENTERED})
    kinds = [meta.at[f, "field_type"] for f in inputs]
    if kinds == ["radio"] * len(inputs) and len(inputs) > 1:  # Scale 4: sum of the six 0-3 ratings
        return work[inputs].apply(pd.to_numeric, errors="coerce").sum(axis=1, min_count=1)
    if kinds == ["radio"]:  # Scale 20: highest number circled
        return pd.to_numeric(work[inputs[0]], errors="coerce")
    box = inputs[kinds.index("checkbox")]
    options = choices(box)
    only = re.search(r"\(([^;)]*?) only", meta.at[SCALE[key], "label"])  # "(Balloon and Books only ...)"
    if only:
        names = re.split(r",\s*|\s+and\s+", only.group(1))
        return ticked(box, [c for c, label in options.items() if any(n.lower() in label.lower() for n in names)])
    older = [c for c, label in options.items() if "older entries" in label]  # Scale 11: /p/ counts with /b/ or /p/
    if older:
        groups = {c: [c] for c in options if c not in older}
        for code in older:
            sound = re.search(r"/\w+/", options[code]).group()
            groups[next(c for c in groups if sound in options[c])].append(code)
        return sum(work[[f"{box}___{c}" for c in group]].eq("1").any(axis=1).astype(int) for group in groups.values())
    count = ticked(box, list(options))
    for other in [f for f, kind in zip(inputs, kinds) if kind == "text"]:  # Scale 17: +1 for a written "other action"
        count = count + work[other].fillna("").str.strip().ne("").astype(int)
    return count


def round_half_up(x):
    """REDCap round(x, 0) for these non-negative scores: .5 always rounds up."""
    return np.floor(x + 0.5 + 1e-9)


rescored = {field: rescore_scale(key) for key, field in calc_scales.items()}
scales = {key: rescored.get(field, work[field]) for key, field in SCALE.items()}  # typed counts are used as entered
clusters = {}
for name, (weights, rounded) in CLUSTERS.items():
    raw = sum(scales[s] * w for s, w in weights.items())  # blank when any of its scales is blank, as in REDCap
    clusters[name] = rescored[CLUSTER[name]] = round_half_up(raw) if rounded else raw
for name, parts in COMPOSITES.items():
    rescored[COMPOSITE[name]] = sum(clusters[c] for c in parts)
rescored[TOTAL] = sum(rescored[COMPOSITE[name]] for name in COMPOSITES)

agree = pd.DataFrame({f: (work[f] == v) | (work[f].isna() & v.isna()) for f, v in rescored.items()})
work["rescore_mismatch"] = ~agree.all(axis=1)
comparison = pd.DataFrame(
    {
        "Score": [ROLE_OF[f] for f in rescored],
        "Field": list(rescored),
        "Visits with a value": [int(work[f].notna().sum()) for f in rescored],
        "Mismatches": [int((~agree[f]).sum()) for f in rescored],
    }
)
n_mismatch = int(comparison["Mismatches"].sum())
check("REDCap scores = re-scored from worksheet items",
      f"{agree.values.sum()} of {agree.size} values agree ({len(rescored)} scores x {len(work)} visits)",
      "OK" if n_mismatch == 0 else "REVIEW")
if n_mismatch:
    print(f"WARNING: {n_mismatch} stored values differ from the re-score; the visits are flagged in the audit file.")
display(comparison)

,Score,Field,Visits with a value,Mismatches
0,Scale 1,csbsbs_scale1,231,0
1,Scale 2,csbsbs_scale2,231,0
2,Scale 3,csbsbs_scale3,231,0
3,Scale 4,csbsbs_scale4,230,0
4,Scale 5,csbsbs_scale5,231,0
5,Scale 6,csbsbs_scale6,231,0
6,Scale 7,csbsbs_scale7,231,0
7,Scale 8,csbsbs_scale8,231,0
8,Scale 9,csbsbs_scale9,231,0
9,Scale 10,csbsbs_scale10,231,0


## 4. Diagnostic group

The group comes from the `demo_status` checkbox at Consent: TD Sib → **TD**, ASD Sib → **ASIB**, PT → **PT**. A child ticked as both ASD Sib and PT is reported as **ASIB + PT** rather than counted in two groups.

PID 5484 leaves `demo_status` blank for most children, so those children take the same field from NANO Study Surveys (PID 4218), matched on child ID. Where both projects have a value, the notebook reports whether they agree. Set `FILL_GROUP_FROM_NANO_SURVEYS = False` in the first cell to use PID 5484 only.

In [5]:
GROUP_RULES = [(r"^ASD", "ASIB"), (r"^TD", "TD"), (r"^PT\b|preterm", "PT"), (r"^Other", "Other")]
GROUP_ORDER = ["ASIB", "ASIB + PT", "PT", "TD", "Other", "Unknown"]
TO_GROUP = {code: next((g for p, g in GROUP_RULES if re.search(p, label, re.I)), label)
            for code, label in choices(STATUS).items()}


def status_groups(frame, id_field):
    """Child ID -> group from the demo_status checkbox; several ticked choices are joined ('ASIB + PT')."""
    groups = {}
    for _, row in frame.iterrows():
        ticked_groups = sorted({TO_GROUP[code] for code in TO_GROUP if row.get(f"{STATUS}___{code}") == "1"})
        if ticked_groups:
            groups[row[id_field]] = " + ".join(ticked_groups)
    return pd.Series(groups, dtype="object")


group_5484 = status_groups(rows[(rows["redcap_event_name"] == STATUS_EVENT) & rows["dde_copy"].isna()], RECORD_ID)
group_4218 = pd.Series(dtype="object")
if FILL_GROUP_FROM_NANO_SURVEYS:
    nano_token, nano_project = connect(NANO_SURVEYS)
    nano_meta = pd.DataFrame(redcap(nano_token, "metadata")).set_index("field_name")
    if STATUS not in nano_meta.index or parse_choices(nano_meta.at[STATUS, "select_choices_or_calculations"]) != choices(STATUS):
        raise ValueError(f"PID {NANO_SURVEYS['pid']} has no {STATUS} field with the same choices as PID {STUDY['pid']}.")
    nano_forms = pd.DataFrame(redcap(nano_token, "formEventMapping"))
    nano_event = nano_forms.loc[nano_forms["form"] == nano_meta.at[STATUS, "form_name"], "unique_event_name"].iloc[0]
    nano_id = nano_meta.index[0]
    nano_rows = pd.DataFrame(redcap(nano_token, "record", type="flat", rawOrLabel="raw",
                                    fields=[nano_id, STATUS], events=[nano_event]))
    group_4218 = status_groups(nano_rows, nano_id)
    print(f"Group fallback: PID {nano_project['project_id']} {nano_project['project_title']}, {len(group_4218)} children with a status")

children = pd.Index(work["child_id"].unique(), name="child_id")
groups = pd.DataFrame({"Diagnostic Group": group_5484.reindex(children)}, index=children)
groups["Group source"] = np.where(groups["Diagnostic Group"].notna(), f"PID {STUDY['pid']} {STATUS}", "not recorded")
fill = groups["Diagnostic Group"].isna() & children.isin(group_4218.index)
groups.loc[fill, "Diagnostic Group"] = group_4218.reindex(children[fill]).to_numpy()
groups.loc[fill, "Group source"] = f"PID {NANO_SURVEYS['pid']} {STATUS}"
groups["Diagnostic Group"] = groups["Diagnostic Group"].fillna("Unknown")
work = work.join(groups, on="child_id")

both = group_5484.index.intersection(group_4218.index).intersection(children)
n_agree = int((group_5484[both] == group_4218[both]).sum())
n_unknown = int((groups["Diagnostic Group"] == "Unknown").sum())
n_dual = int(groups["Diagnostic Group"].str.contains(r"\+").sum())
check(f"Diagnostic group from PID {STUDY['pid']} {STATUS}", f"{int((groups['Group source'] == f'PID {STUDY['pid']} {STATUS}').sum())} children", "INFO")
if FILL_GROUP_FROM_NANO_SURVEYS:
    check(f"Diagnostic group filled from NANO Study Surveys (PID {NANO_SURVEYS['pid']})",
          f"{int(fill.sum())} children; the two projects agree for {n_agree} of {len(both)} children recorded in both",
          "INFO" if n_agree == len(both) else "REVIEW")
check("Children with no diagnostic group", f"{n_unknown} children (reported as Unknown)", "REVIEW" if n_unknown else "OK")
check("Children ticked in two groups", f"{n_dual} children, reported as their own group (e.g. ASIB + PT)", "INFO")
display(pd.crosstab(groups["Diagnostic Group"], groups["Group source"], margins=True))

Group fallback: PID 4218 NANO Study Surveys, 222 children with a status


Group source,PID 4218 demo_status,PID 5484 demo_status,All
Diagnostic Group,,,
ASIB,32,16,48
ASIB + PT,4,0,4
PT,44,1,45
TD,39,6,45
All,119,23,142


## 5. Examiner names

`Scored by` is typed by hand, so one person appears under several spellings (`CC`, `Caroline`, `Caroline christianson`). Each value is matched to the lab roster in this order:

1. exact full name (any capitalisation, extra spaces ignored)
2. initials: `CC` → Caroline Christianson, `RLM` → Ramiro Lucas-Mariano
3. close spelling: `Makenzie Sotos` → Makenzie Soto
4. first name, with a last initial if typed: `Katerri` → Katerri Pringle, `Lauren P` → Lauren Puttock

A value that matches no one is kept as typed and flagged in the table below. A blank is reported as **Not recorded**. Add new scorers to `ROSTER`.

In [6]:
ROSTER = ["Aryana Hughey", "Axie Acosta", "Caroline Christianson", "Cecilia Thomas", "Eilis McLaughlin",
          "Elizabeth Dixon", "Emma Platt", "Katerri Pringle", "Katie Rowland", "Lauren Puttock",
          "Makenzie Soto", "Ramiro Lucas-Mariano", "Sanjana Oak", "Tessa Djiko"]
TYPO_ALIASES = {"RML": "Ramiro Lucas-Mariano"}  # initials typed out of order

by_initials = {}
for person in ROSTER:
    by_initials.setdefault("".join(part[0] for part in re.split(r"[\s-]+", person)).upper(), []).append(person)
INITIALS = {k: v[0] for k, v in by_initials.items() if len(v) == 1} | TYPO_ALIASES
FULL_NAME = {person.lower(): person for person in ROSTER}


def standardise_examiner(raw):
    """Return (examiner, how it was matched) for one typed 'Scored by' value."""
    text = re.sub(r"\s+", " ", str(raw)).strip().rstrip(".")
    if not text:
        return "Not recorded", "blank in REDCap"
    if text.lower() in FULL_NAME:
        return FULL_NAME[text.lower()], "exact name"
    if " " not in text and text.upper() in INITIALS:
        return INITIALS[text.upper()], "initials"
    close = difflib.get_close_matches(text.lower(), list(FULL_NAME), n=1, cutoff=0.85)
    if close:
        return FULL_NAME[close[0]], "close spelling"
    first, _, last = text.partition(" ")
    hits = [p for p in ROSTER
            if difflib.SequenceMatcher(None, first.lower(), p.split()[0].lower()).ratio() >= 0.85
            and p.split(" ", 1)[1].lower().startswith(last.lower())]
    if len(hits) == 1:
        return hits[0], "first name + last initial" if last else "first name"
    return text, "NOT ON ROSTER: check"


matched = work[EXAMINER].map(standardise_examiner)
work["Examiner"] = matched.str[0]
work["Examiner match"] = matched.str[1]
examiner_map = (
    work.assign(**{"Typed in REDCap": "'" + work[EXAMINER] + "'"})
    .groupby(["Examiner", "Typed in REDCap", "Examiner match"]).size().rename("Visits").reset_index()
    .sort_values(["Examiner", "Visits"], ascending=[True, False], ignore_index=True)
)
recorded = work["Examiner"] != "Not recorded"
n_blank, n_blank_scored = int((~recorded).sum()), int((~recorded & work["scored"]).sum())
n_off_roster = int(work["Examiner match"].str.startswith("NOT").sum())
check("Examiner names standardised",
      f"{work.loc[recorded, EXAMINER].nunique()} typed spellings -> {work.loc[recorded, 'Examiner'].nunique()} people", "INFO")
check("Examiner not recorded in REDCap",
      f"{n_blank} visits ({n_blank_scored} scored) have a blank 'Scored by'; shown as 'Not recorded'", "REVIEW" if n_blank else "OK")
check("Examiner names not on the roster", f"{n_off_roster} visits, kept as typed", "REVIEW" if n_off_roster else "OK")
display(examiner_map)

,Examiner,Typed in REDCap,Examiner match,Visits
0,Axie Acosta,'Axie Acosta',exact name,2
1,Caroline Christianson,'Caroline Christianson',exact name,58
2,Caroline Christianson,'CC',initials,3
3,Caroline Christianson,'Caroline christianson',exact name,1
4,Caroline Christianson,'Caroline',first name,1
5,Cecilia Thomas,'Cecilia Thomas',exact name,10
6,Eilis McLaughlin,'Eilis McLaughlin',exact name,6
7,Elizabeth Dixon,'Elizabeth Dixon',exact name,3
8,Emma Platt,'Emma Platt',exact name,32
9,Katerri Pringle,'Katerri Pringle',exact name,34


## 6. Metrics by examiner, visit and diagnostic group

For each examiner × visit × group, over its N scored visits:

- **Mean** = Σx ÷ N
- **SD** = √( Σ(x − mean)² ÷ (N − 1) ), the sample standard deviation (Excel `STDEV.S`)

Worked example: Social scores 18, 21 and 24 give mean 21 and SD = √((9 + 0 + 9) ÷ 2) = **3.00**. A single visit (N = 1) has no spread to measure, so its SD is shown as **N/A** rather than a blank or a misleading 0.

Every scored visit lands in exactly one row, so the N column adds up to the number of scored visits.

In [7]:
used = work[work["scored"] & ~(EXCLUDE_REDCAP_FLAGGED & work[NEEDS_REVIEW].eq("1"))]
used = used.rename(columns={field: name for name, field in COMPOSITE.items()})
n_flagged = int(used[NEEDS_REVIEW].eq("1").sum())

summary = (
    used.groupby(["Examiner", "Visit", "Diagnostic Group"])
    .agg(**{f"{name}_{stat}": (name, "mean" if stat == "Mean" else "std") for name in COMPOSITES for stat in ("Mean", "SD")},
         N=("Social", "size"))
    .reset_index()
)
summary.insert(1, "Study", "CSBS-BS")
sd_columns = [f"{name}_SD" for name in COMPOSITES]
assert summary.loc[summary["N"] > 1, sd_columns].notna().all().all(), "SD is missing for a row with N > 1"
summary[sd_columns] = summary[sd_columns].astype(object).where(summary["N"] > 1, "N/A")

summary = (
    summary.assign(
        _group=summary["Diagnostic Group"].map({g: i for i, g in enumerate(GROUP_ORDER)}).fillna(len(GROUP_ORDER)),
        _unrecorded=summary["Examiner"].eq("Not recorded"),
        _visit=summary["Visit"].map(VISIT_ORDER.index),
    )
    .sort_values(["_group", "Diagnostic Group", "_unrecorded", "Examiner", "_visit"])
    .loc[:, ["Examiner", "Study", "Visit", "Diagnostic Group", "Social_Mean", "Social_SD",
             "Speech_Mean", "Speech_SD", "Symbolic_Mean", "Symbolic_SD", "N"]]
    .reset_index(drop=True)
)

assert summary["N"].sum() == len(used), "N does not add up to the number of scored visits"
assert not summary.isna().any().any() and not summary.astype(str).apply(lambda c: c.str.strip().eq("")).any().any(), "blank cell"
check("Visits flagged 'Check before using these scores' in REDCap",
      f"{n_flagged} visits {'left out' if EXCLUDE_REDCAP_FLAGGED else 'kept (set EXCLUDE_REDCAP_FLAGGED to drop them)'}",
      "REVIEW" if n_flagged else "OK")
check("Metrics table", f"{len(summary)} rows; N adds up to {summary['N'].sum()} = {len(used)} scored visits; no blank cells")
check("SD definition", "Sample standard deviation (n - 1); N/A when N = 1 because one visit has no spread", "INFO")
display(summary.map(lambda v: f"{v:.2f}" if isinstance(v, float) else v))

,Examiner,Study,Visit,Diagnostic Group,Social_Mean,Social_SD,Speech_Mean,Speech_SD,Symbolic_Mean,Symbolic_SD,N
0,Axie Acosta,CSBS-BS,12 Months,ASIB,30.00,N/A,0.00,N/A,15.00,N/A,1
1,Caroline Christianson,CSBS-BS,9 Months,ASIB,16.17,6.68,1.33,2.16,2.17,2.56,6
2,Caroline Christianson,CSBS-BS,12 Months,ASIB,31.44,12.14,3.00,3.71,7.22,3.73,9
3,Cecilia Thomas,CSBS-BS,9 Months,ASIB,18.00,N/A,0.00,N/A,5.00,N/A,1
4,Cecilia Thomas,CSBS-BS,12 Months,ASIB,27.00,9.90,6.50,3.54,7.50,7.78,2
5,Eilis McLaughlin,CSBS-BS,9 Months,ASIB,18.75,4.03,1.50,1.73,8.50,1.29,4
6,Eilis McLaughlin,CSBS-BS,12 Months,ASIB,26.00,N/A,0.00,N/A,21.00,N/A,1
7,Elizabeth Dixon,CSBS-BS,9 Months,ASIB,19.50,2.12,0.00,0.00,4.50,0.71,2
8,Emma Platt,CSBS-BS,9 Months,ASIB,16.00,6.35,0.86,1.46,5.29,1.98,7
9,Emma Platt,CSBS-BS,12 Months,ASIB,16.60,11.24,1.00,1.41,6.40,4.10,5


## 7. Save the workbook

- **Metrics**: the table above (same columns as before).
- **QC Checks**: every check this run made, with OK / REVIEW / INFO.
- **Field Map**: the REDCap variable, live label and formula behind every number.

The visit-level audit (child ID, examiner as typed, group source, scores, reason left out) goes to `IPSA/csbsbs_audit_secure/`, which is git-ignored because it is participant-level.

In [8]:
qc_table = pd.DataFrame(qc)

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="Metrics", index=False)
    qc_table.to_excel(writer, sheet_name="QC Checks", index=False)
    field_map.to_excel(writer, sheet_name="Field Map", index=False)
    for sheet in writer.book.worksheets:
        sheet.freeze_panes = "A2"
        for cell in sheet[1]:
            cell.font = Font(bold=True)
        for column in sheet.columns:
            width = max(len(str(cell.value)) for cell in column if cell.value is not None)
            sheet.column_dimensions[column[0].column_letter].width = min(max(width + 2, 10), 70)
    for row in writer.book["Metrics"].iter_rows(min_row=2):
        for cell in row:
            if isinstance(cell.value, float):
                cell.number_format = "0.00"
            elif cell.value == "N/A":
                cell.alignment = Alignment(horizontal="right")

audit = work.rename(columns={field: name for name, field in COMPOSITE.items()})
audit = audit[[RECORD_ID, "Visit", EXAMINER, "Examiner", "Examiner match", "Diagnostic Group", "Group source",
               *COMPOSITES, TOTAL, NEEDS_REVIEW, "rescore_mismatch", "scored"]].rename(
    columns={EXAMINER: "Scored by (typed)", TOTAL: "Total", NEEDS_REVIEW: "REDCap check flag",
             "rescore_mismatch": "Re-score differs from REDCap"})
audit["Used in metrics"] = np.where(audit.index.isin(used.index), "yes",
                                    np.where(audit["scored"], "no: REDCap check flag", "no: a composite is blank"))
audit = pd.concat([audit.drop(columns="scored"),
                   pending.rename(columns={"child_id": RECORD_ID}).drop(columns="redcap_event_name")
                   .assign(**{"Used in metrics": "no: scored only in a DDE copy, not merged yet"})], ignore_index=True)
AUDIT_CSV.parent.mkdir(exist_ok=True)
audit.to_csv(AUDIT_CSV, index=False)

print(f"Saved {OUTPUT_XLSX.relative_to(REPO_ROOT)} (Metrics, QC Checks, Field Map)")
print(f"Saved {AUDIT_CSV.relative_to(REPO_ROOT)} ({len(audit)} visits, git-ignored)\n")
print(f"- {len(used)} scored visits from {used['child_id'].nunique()} children at "
      f"{', '.join(sorted(used['Visit'].unique(), key=VISIT_ORDER.index))}, scored by "
      f"{used.loc[used['Examiner'] != 'Not recorded', 'Examiner'].nunique()} examiners "
      f"({int((used['Examiner'] == 'Not recorded').sum())} visits with no examiner recorded).")
print(f"- Every REDCap score was re-derived from the worksheet: {n_mismatch} mismatches.")
print(f"- Left out: {n_partial} partly scored forms, {len(pending)} visits not merged yet.")
print(f"- {(qc_table['Status'] == 'REVIEW').sum()} checks need a look in REDCap (Status = REVIEW):")
display(qc_table)

Saved IPSA/csbsbs_metrics_by_examiner.xlsx (Metrics, QC Checks, Field Map)
Saved IPSA/csbsbs_audit_secure/csbsbs_visit_audit.csv (263 visits, git-ignored)

- 213 scored visits from 139 children at 9 Months, 12 Months, scored by 13 examiners (10 visits with no examiner recorded).
- Every REDCap score was re-derived from the worksheet: 0 mismatches.
- Left out: 18 partly scored forms, 32 visits not merged yet.
- 4 checks need a look in REDCap (Status = REVIEW):


,Check,Result,Status
0,REDCap project,PID 5484: NANO Lab Assessments & Double Data Entry (token TOKEN_NA...,OK
1,CSBS BS instrument,"csbs_bs ('CSBS BS') at 9 Months, 12 Months",OK
2,Fields found by label,37 fields; every label matched exactly one field,OK
3,Score formulas read the right inputs,"11 of 11 cluster, composite and total formulas",OK
4,Double data entry copies left out,544 rows (--1 / --2); each visit is read from its merged record,INFO
5,Visits with CSBS BS worksheet data (merged records),"231 visits, 142 children",INFO
6,Visits scored on all three composites,213 visits,OK
7,Partly scored visits left out (a composite is blank),18 visits: finish the worksheet in REDCap,REVIEW
8,"Visits scored only in a DDE copy, not merged yet (left out)",32 visits: merge them in the Data Comparison Tool,REVIEW
9,Legacy CSBS BS instrument,0 visits hold data on csbs_bs_old,OK
